In [0]:
# Import libraries
import dlt
from pyspark import pipelines as dp
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
@dlt.view
def silver_employee_cdf_v2():
    df = spark.readStream.table("LIVE.bronze_employee_cdf")
 
    # cast data types
    df = df.withColumn("Employee_ID", df["Employee_ID"].cast("string"))
    df = df.withColumn("Performance_Score", df["Performance_Score"].cast("int"))
    df = df.withColumn("Monthly_Salary", df["Monthly_Salary"].cast("double"))
 
    # trim text fields
    df = df.withColumn("Name", trim(col("Name")))
    df = df.withColumn("Department", trim(col("Department")))
 
    # add row status column
    df = df.withColumn("row_status", lit("New"))
 
    return df

In [0]:
dlt.create_streaming_table("employee_cdf_type2_stage")
 
dlt.apply_changes(

    target = "employee_cdf_type2_stage",

    source = "silver_employee_cdf_v2",

    keys = ["Employee_ID"],

    sequence_by = struct("Employee_ID", "_commit_timestamp"),

    ignore_null_updates = True,             # columns with null retain existing values

    apply_as_deletes = expr("_change_type = 'delete'"),  # handle deletions

    #except_column_list = ["operation", "sequenceNum"],  # optional exclusions

    stored_as_scd_type = 2                  # Type 2 = track historical changes

)
 